<a href="https://colab.research.google.com/github/gaga-zhou/v_1/blob/main/caj2pdf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## CAJ 文件批量转 PDF 教程

这个 Colab Notebook 可以帮助你将 `.caj` 格式的文献批量转换为 `.pdf` 格式。

**使用步骤：**

1.  **上传 CAJ 文件：**
    *   在左侧文件浏览器中，找到 `/content` 目录。
    *   右键点击 `/content`，选择“新建文件夹”，命名为 `caj` (如果 `/content/caj` 文件夹已存在，则跳过此步)。
    *   右键点击新建的 `/content/caj` 文件夹，选择“上传”。
    *   选择你所有要转换的 `.caj` 文件并上传。
    *   **重要提示：请耐心等待所有文件上传完成，确保文件上传进度条消失，所有文件都显示在 `/content/caj` 文件夹中。**

2.  **运行所有代码单元格：**
    *   在顶部菜单栏中，点击“运行时 (Runtime)” -> “运行所有单元格 (Run all)”。

3.  **下载转换后的 PDF 文件：**
    *   当所有单元格运行完毕后，一个名为 `converted_pdfs.zip` 的压缩包会自动下载到你的本地电脑。
    *   这个压缩包包含了所有转换成功的 PDF 文件。

---

**运行时补充说明：**

*   此笔记本默认使用 Colab 提供的 **CPU 运行时环境**进行文件转换和处理。通常无需额外配置。
*   对于大量的 `.caj` 文件，转换过程可能需要一些时间，具体取决于文件大小和数量。

In [1]:
# 1. 安装系统依赖
!apt-get install -y mupdf-tools

# 2. 克隆源码并安装 Python 依赖
!rm -rf caj2pdf
!git clone --depth 1 https://github.com/caj2pdf/caj2pdf.git
!pip install pikepdf PyPDF2

# 3. 将脚本目录添加到环境变量，或创建软链接以便直接调用
import os
os.environ['PATH'] += ":" + os.path.abspath("caj2pdf")
!chmod +x caj2pdf/caj2pdf
!ln -sf $(pwd)/caj2pdf/caj2pdf /usr/local/bin/caj2pdf

# 4. 验证
!caj2pdf --version

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libgumbo1 libjbig2dec0 libmujs1
The following NEW packages will be installed:
  libgumbo1 libjbig2dec0 libmujs1 mupdf-tools
0 upgraded, 4 newly installed, 0 to remove and 3 not upgraded.
Need to get 45.7 MB of archives.
After this operation, 73.4 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libgumbo1 amd64 0.10.1+dfsg-2.4 [106 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libjbig2dec0 amd64 0.19-3build2 [64.7 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libmujs1 amd64 1.1.3-3 [117 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 mupdf-tools amd64 1.19.0+ds1-2 [45.4 MB]
Fetched 45.7 MB in 1s (58.7 MB/s)
Selecting previously unselected package libgumbo1:amd64.
(Reading database ... 118252 files and directories currently installed.)
Pre

In [2]:
import os
import glob

# 定义输入和输出路径
input_dir = '/content/caj'
output_dir = '/content/pdf_output'

# 创建输出目录
os.makedirs(output_dir, exist_ok=True)

# 获取所有 .caj 文件
caj_files = glob.glob(os.path.join(input_dir, '*.caj'))

print(f'找到 {len(caj_files)} 个 CAJ 文件，准备转换...')

for caj_path in caj_files:
    filename = os.path.basename(caj_path)
    pdf_name = filename.rsplit('.', 1)[0] + '.pdf'
    pdf_path = os.path.join(output_dir, pdf_name)

    print(f'正在转换: {filename} -> {pdf_name}')
    # 修改调用方式：指定 convert 后的输出格式为 pdf
    !caj2pdf convert "{caj_path}" -o "{pdf_path}"

print('\n转换完成！文件保存在:', output_dir)

找到 0 个 CAJ 文件，准备转换...

转换完成！文件保存在: /content/pdf_output


In [3]:
# Patching cajparser.py to handle empty TOC page numbers
import os

file_path = '/content/caj2pdf/cajparser.py'
with open(file_path, 'r') as f:
    lines = f.readlines()

with open(file_path, 'w') as f:
    for line in lines:
        # Look for the line: page = int(toc_bytes[2][0:pg_end])
        if 'page = int(toc_bytes[2][0:pg_end])' in line:
            indent = line[:line.find('page')]
            f.write(f'{indent}try:\n')
            f.write(f'{indent}    page = int(toc_bytes[2][0:pg_end])\n')
            f.write(f'{indent}except (ValueError, IndexError):\n')
            f.write(f'{indent}    page = 0\n')
        else:
            f.write(line)

print('Successfully patched cajparser.py to ignore malformed TOC entries.')

Successfully patched cajparser.py to ignore malformed TOC entries.


In [4]:
import shutil
from google.colab import files

# 定义压缩文件的路径（不含扩展名）
zip_filename = '/content/converted_pdfs'

# 将 output_dir 目录压缩为 zip
shutil.make_archive(zip_filename, 'zip', '/content/pdf_output')

# 下载到本地
files.download(zip_filename + '.zip')

print(f'已创建压缩包并尝试下载: {zip_filename}.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

已创建压缩包并尝试下载: /content/converted_pdfs.zip
